# 06 — Structured Streaming in Fabric Spark

Covers file-based streaming ("autoloader-style" incremental file ingestion), Eventhouse/Event Hub
style streaming sources, checkpointing, triggers, and the `foreachBatch` + `MERGE` pattern used to
build streaming upserts into a Delta table.


## 1. Incremental file ingestion (readStream over a folder)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

txn_schema = StructType([
    StructField("txn_id", StringType()),
    StructField("account_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("txn_ts", TimestampType()),
])

df_stream_raw = (
    spark.readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 10)      # controls micro-batch size
    .json("Files/streaming_landing/transactions/")
)


## 2. Transform the stream just like a batch DataFrame

In [ ]:
from pyspark.sql import functions as F

df_stream_clean = (
    df_stream_raw
    .filter(F.col("amount").isNotNull())
    .withColumn("ingest_ts", F.current_timestamp())
    .withWatermark("txn_ts", "10 minutes")   # bounds late-arriving data / stateful ops
)


## 3. Simple append sink: write straight to a Delta table

In [ ]:
query = (
    df_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/checkpoints/bronze_transactions")
    .trigger(processingTime="1 minute")
    .toTable("bronze_transactions")
)

# query.awaitTermination()   # uncomment to block the cell until the stream stops
# query.stop()               # call from another cell (or on a timer) to stop gracefully


## 4. `foreachBatch` + `MERGE`: streaming upserts

The most common real-world pattern — dedupe/upsert each micro-batch into a Delta table instead
of blindly appending.

In [ ]:
from delta.tables import DeltaTable

def upsert_to_delta(microbatch_df, batch_id):
    if not spark.catalog.tableExists("silver_transactions"):
        microbatch_df.write.format("delta").saveAsTable("silver_transactions")
        return

    target = DeltaTable.forName(spark, "silver_transactions")
    (
        target.alias("t")
        .merge(microbatch_df.alias("s"), "t.txn_id = s.txn_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_stream_clean.writeStream
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", "Files/checkpoints/silver_transactions")
    .trigger(processingTime="2 minutes")
    .start()
)


## 5. Trigger types

In [ ]:
# Micro-batch on a fixed interval (most common for scheduled/near-real-time loads)
.trigger(processingTime="30 seconds")

# Process everything currently available then STOP — great for a Fabric pipeline that runs
# the notebook on a schedule instead of keeping a stream running 24/7
.trigger(availableNow=True)

# One-shot micro-batch (mainly for testing)
.trigger(once=True)


## 6. Reading from Eventhouse / Kafka-compatible Event Hubs

When the source is an Eventstream / Event Hub (common for banking transaction feeds or IoT
telemetry), use the Kafka-compatible endpoint.

In [ ]:
eh_namespace = "my-eventhub-namespace"
eh_name = "transactions-eh"
connection_string = mssparkutils.credentials.getSecret(
    "https://<kv-name>.vault.azure.net/", "eventhub-connection-string"
)

kafka_options = {
    "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
    "subscribe": eh_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="$ConnectionString" password="{connection_string}";'
    ),
    "startingOffsets": "latest",
}

df_eh_raw = spark.readStream.format("kafka").options(**kafka_options).load()

df_eh_parsed = (
    df_eh_raw
    .selectExpr("CAST(value AS STRING) AS json_payload", "timestamp AS eh_enqueued_ts")
    .select(F.from_json("json_payload", txn_schema).alias("data"), "eh_enqueued_ts")
    .select("data.*", "eh_enqueued_ts")
)


## 7. Monitoring & stopping streams safely

In [ ]:
for s in spark.streams.active:
    print(s.id, s.name, s.status)

# Graceful shutdown pattern for a scheduled notebook using availableNow
query.awaitTermination()
print("Batch finished. Last progress:", query.lastProgress)


Next notebook: **07 — Real-World Medallion ETL (Banking)**, which ties ingestion,
Delta MERGE, and orchestration together into one end-to-end pipeline.